# Lab 21 — Fine-tuning LLMs · RUN ALL (Kaggle / Colab)

Kaggle: bật Internet và chọn GPU T4×2 trong Notebook Settings. Colab: chọn T4 GPU.
Chạy các ô từ trên xuống. Bài nộp chạy full eval (`EVAL_LIMIT=""`); pipeline NB1–NB5 thường mất khoảng 100–130 phút trên T4.

| Ô | Nội dung |
|---|---|
| 1 | Clone fork + cài dependencies |
| 2 | Smoke: imports, dữ liệu, unit checks |
| 3 | Core pipeline NB1 → NB5 |
| 4 | Gatekeeper + in kết quả |
| 5 | Tạo ZIP toàn repo và upload Hugging Face (Kaggle Secret `HF_TOKEN`) |


In [ ]:
# @title 1. Setup — clone fork + install
from pathlib import Path
import os, subprocess, sys
from importlib.metadata import version

REPO = "https://github.com/MinMinhMin/Day21-Track3-Finetuning-Lab"
if (Path.cwd() / "scripts" / "colab_run.py").is_file():
    REPO_DIR = Path.cwd()
    if (REPO_DIR / ".git").exists():
        subprocess.run(["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", REPO], check=False)
else:
    if Path("/kaggle/working").is_dir():
        WORK_DIR = Path("/kaggle/working")
    elif Path("/content").is_dir():
        WORK_DIR = Path("/content")
    else:
        WORK_DIR = Path.cwd()
    REPO_DIR = WORK_DIR / "Day21-Track3-Finetuning-Lab"
    if REPO_DIR.exists():
        if not (REPO_DIR / ".git").exists():
            raise RuntimeError(f"Đã có thư mục nhưng không phải git repo: {REPO_DIR}")
        subprocess.run(["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", REPO], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)
    else:
        subprocess.run(["git", "clone", "-q", REPO, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR / "src"))

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
               check=True)

os.environ.setdefault("COMPUTE_TIER", "T4")
import torch
print("commit:", subprocess.run(["git", "rev-parse", "--short", "HEAD"],
                                 capture_output=True, text=True, cwd=REPO_DIR).stdout.strip())
print("repo:", REPO_DIR)
print("stack:", {name: version(name) for name in
                  ("torch", "transformers", "trl", "peft", "accelerate", "tokenizers")})
gpu_names = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
print("GPU(s):", gpu_names if gpu_names else "NONE — bật GPU trong Notebook Settings")


In [ ]:
# @title 2. Smoke — imports, seed data, unit tests (no GPU needed)
import subprocess, sys
subprocess.run([sys.executable, "scripts/verify.py", "--smoke"], check=True)


In [ ]:
# @title 3. Core pipeline — NB1 → NB5
# Sau bản sửa FP16: session mới, FORCE_RETRAIN=True, chạy lại NB1–NB5.
# EVAL_LIMIT truncates both eval sets: "" = full run (submittable),
# 8 = ~fast smoke pass. STAGES lets you resume after a failure.
import os, subprocess, sys
COMPUTE_TIER = "T4"        # @param ["CPU","LAPTOP","T4","BIGGPU"]
EVAL_LIMIT   = ""         # @param ["", "4", "8", "16", "25"]
STAGES       = "nb1 nb2 nb3 nb4 nb5"   # @param {type:"string"}
FORCE_RETRAIN = True                    # @param {type:"boolean"}

os.environ["COMPUTE_TIER"] = COMPUTE_TIER
if FORCE_RETRAIN:
    os.environ["FORCE_RETRAIN"] = "1"
else:
    os.environ.pop("FORCE_RETRAIN", None)
if EVAL_LIMIT:
    os.environ["EVAL_LIMIT"] = EVAL_LIMIT
else:
    os.environ.pop("EVAL_LIMIT", None)

from labkit import device
print(device.banner(), "\n")

subprocess.run([sys.executable, "-u", "scripts/colab_run.py", *STAGES.split()],
               check=True)


In [ ]:
# @title 4. Gatekeeper + results
!python scripts/verify.py
print("\n================ results/ ================")
!ls -la results/
!echo && echo "---- runs.csv ----" && cat results/runs.csv 2>/dev/null
!echo && echo "---- verdict.json ----" && cat results/verdict.json 2>/dev/null


## 5. Đóng gói repo và upload Hugging Face

Trước khi chạy ô này trên Kaggle, tạo Secret `HF_TOKEN` có quyền ghi vào repo `MinMinMinMin/KL` (Notebook → Add-ons → Secrets). ZIP gồm mã nguồn, dữ liệu, notebook, adapter và `results/`; bỏ `.env`, `__pycache__`, token, Git history và cache môi trường. ZIP được ghi ở `/kaggle/working`, ngoài thư mục repo.


In [ ]:
# @title 5. Zip toàn repo (loại secrets/cache) và upload Hugging Face
from pathlib import Path
import os
import zipfile

from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi, login

HF_REPO_ID = "MinMinMinMin/KL"
HF_REPO_TYPE = "model"
HF_REPO_SUBDIR = "lab21_repo"

ROOT = Path.cwd()
if not (ROOT / "results").is_dir():
    raise FileNotFoundError(f"Không thấy results/ trong repo: {ROOT}")

hf_token = UserSecretsClient().get_secret("HF_TOKEN")
if not hf_token:
    raise RuntimeError("Kaggle Secret 'HF_TOKEN' đang rỗng.")

workspace = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else ROOT.parent
zip_path = workspace / f"{ROOT.name}_results.zip"
excluded_dirs = {
    ".git", "__pycache__", ".venv", "venv", ".ipynb_checkpoints",
    ".pytest_cache", ".cache", "hf_cache", ".aws",
}
file_count = 0
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for current, dirnames, filenames in os.walk(ROOT):
        dirnames[:] = [name for name in dirnames if name not in excluded_dirs]
        for filename in filenames:
            source = Path(current) / filename
            if source.resolve() == zip_path.resolve():
                continue
            if filename == ".env" or (filename.startswith(".env.") and filename != ".env.example"):
                continue
            if filename.endswith((".pyc", ".pyo", ".token")):
                continue
            relative = source.relative_to(ROOT)
            archive.write(source, (Path(ROOT.name) / relative).as_posix())
            file_count += 1

# Verify excluded paths before uploading.
with zipfile.ZipFile(zip_path) as archive:
    members = [Path(name) for name in archive.namelist()]
    bad_members = [
        member.as_posix() for member in members
        if any(part in excluded_dirs for part in member.parts)
        or member.name == ".env"
        or (member.name.startswith(".env.") and member.name != ".env.example")
        or member.suffix in {".pyc", ".pyo", ".token"}
    ]
if bad_members:
    zip_path.unlink(missing_ok=True)
    raise RuntimeError(f"ZIP chứa file bị loại trừ: {bad_members[:10]}")

print(f"ZIP: {zip_path} ({file_count} files, {zip_path.stat().st_size / 1024**2:.1f} MB)")

try:
    login(token=hf_token, add_to_git_credential=False)
    upload_path = f"{HF_REPO_SUBDIR.strip('/')}/{zip_path.name}"
    HfApi().upload_file(
        path_or_fileobj=str(zip_path),
        path_in_repo=upload_path,
        repo_id=HF_REPO_ID,
        repo_type=HF_REPO_TYPE,
    )
    print("Uploaded:", f"https://huggingface.co/{HF_REPO_ID}/blob/main/{upload_path}")
finally:
    del hf_token
